# Lesson 21 — Spatial Branch-and-Bound

## Learning objectives

1. State the spatial-B&B algorithm for nonconvex MINLP.
2. Recognize the role of FBBT, OBBT, and convex relaxations inside the tree.
3. Read a `discopt` spatial-B&B result: incumbent, dual bound, gap, `gap_certified`, node count.
4. Diagnose when spatial B&B is failing (gap not closing).

## 1. Algorithm

```
Stack = {root box B}
LB = -inf, UB = +inf, incumbent = None
while gap > tol and Stack non-empty:
    pick node N (with box B_N)
    LB_N = solve convex relaxation over B_N
    if relaxation infeasible:        continue   # prune (infeasibility)
    if LB_N >= UB - tol:              continue   # prune (bound)
    if relaxation looks "tight" — auxiliaries near McCormick-tight,
       integer variables near-integer:
        x_local = local_NLP(N)                  # try to recover a primal point
        if x_local feasible: UB <- min(UB, f(x_local))
    if (UB - LB_N) / max(1, |UB|) < tol: continue   # gap closed at this node
    pick branching variable j (most violated, longest edge, etc.)
    split B_N along j -> two children, push
update LB <- min LB_N over open nodes
```

This is the spatial B&B of {cite:p}`Smith1999,Tawarmalani2005,Belotti2009`. The key differences from MILP B&B (Lesson 6) are **branching on continuous variables** (typically the auxiliaries $w_{ij} = x_i x_j$ that linearise nonconvex terms) to refine the convex relaxation, and the `relaxation looks tight` cue for the local-NLP recovery — bare integer feasibility isn't enough since continuous nonconvexities remain.

## 2. Convex relaxation at a node

For each nonconvex term, build the McCormick (Lesson 16) or αBB underestimator over the *current* box. The relaxation is an LP (if all relaxations are linear) or convex NLP, solved cheaply at each node.

This is why **bound tightening** (Lesson 15) is so critical in MINLP — each tightening *retroactively* tightens every relaxation in the tree.

## 3. Branching

Continuous branching variables are picked by:

- **Most violated** — branch on the variable whose envelope inequality is most violated by the LP solution.
- **Longest edge** — bisect the longest box edge.
- **Pseudocost-style** (recent literature).

Where to split (golden ratio? mid-point?): typically at the LP solution value or midpoint, with safeguards.

## 4. Convergence and termination

Spatial B&B is **finite for ε-optimality** but not finite in exact arithmetic for general nonconvex MINLPs. Convergence is by gap squeezing: every infinite branching sequence must shrink some box dimension to zero, eventually making the relaxation exact at the limit point {cite:p}`HorstTuy1996,Floudas1999`.

Practical termination: gap < ε (relative or absolute), or node/time limit.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Nonconvex NLP: pump network design (toy)
m = Model("pump_net")
x = m.continuous("x", lb=0, ub=10)     # flow
y = m.continuous("y", lb=0, ub=10)     # pressure
m.subject_to(x * y >= 6)                  # work (bilinear, nonconvex)
m.subject_to(0.5 * x**2 + y <= 12)        # head loss
m.minimize(0.1 * x**3 + y)                # power cost

r = m.solve()
print(f"status: {r.status}  f* = {r.objective:.6f} at x = {r.x}")
print(f"dual bound = {r.bound:.6f}, gap = {r.gap:.2e}, gap_certified = {r.gap_certified}")
print(f"nodes = {r.node_count}")


## 5. Levers in `discopt`'s spatial B&B

The Python API for `m.solve()` exposes the knobs you actually reach for
when a spatial B&B run is not closing. `solve()` **rejects unknown keyword
arguments** with a `TypeError` (and a near-match suggestion), so a misspelled
option can no longer silently run the default.

| Argument | Effect |
|---|---|
| `time_limit=<sec>` | Wall-clock deadline. On a limit exit you still get the incumbent and an *anytime* dual bound. |
| `gap_tolerance=<eps>` / `abs_gap_tolerance=<eps>` | Relative / absolute gap for termination (either one stops the search). |
| `initial_solution={var: value, ...}` | Warm start: validated, clamped/rounded if needed, and used as an initial incumbent so the bound can prune from node zero. |
| `partitions=k` | Use $k$ piecewise McCormick partitions per nonconvex term instead of the single-piece envelope. Tighter relaxation at $O(k)$ size cost. |
| `obbt_at_root=False`, `presolve=False` | Switch off root OBBT / root FBBT presolve (useful for measuring what bound tightening buys you, Exercise 2). |
| `strategy="best_first"` / `"depth_first"` / `"best_estimate"` | Node-selection rule. (Variable selection is not user-selectable: discopt uses reliability branching — pseudocosts plus strong branching — inside its Rust tree manager.) |
| `nlp_bb=True/False` | Force NLP-BB (convex MINLP) or spatial B&B. Default `None` auto-detects. |
| `skip_convex_check=True` | Skip automatic convexity detection. By default a model proved convex is solved by a single certified NLP call (`r.convex_fast_path == True`, no spatial branching). |
| `solver="amp"` | Adaptive Multivariate Partitioning {cite:p}`Nagarajan2019`: an alternative *global* engine that refines a piecewise relaxation where the gap is largest; often better when spatial B&B stalls. |
| `incumbent_callback`, `lazy_constraints`, `node_callback`, `cut_callback` | Extension hooks for custom acceptance, lazy cuts, per-batch monitoring, and user cuts at every node (user cuts are *validated* against known feasible points before use). |
| `tuning=do.SolverTuning(...)` | Typed, per-call object for advanced relaxation / B&B switches (RLT families, node-bound mode, ...); replaces the old `DISCOPT_*` environment variables. |
| `validate=True` | Run an Examiner-style KKT check on the returned point and attach it as `r.validation_report`. |

Node relaxations are McCormick-type LPs built over the node box (with
RLT/PSD/SOC cuts where structure allows); local NLPs used to find
incumbents are solved by POUNCE, a Rust port of the Ipopt interior-point
method {cite:p}`Wachter2006`. Before relaxing, discopt's factorable
reformulation lifts composite terms (products, fractional powers,
transcendentals) into auxiliary variables, each with its own envelope —
exactly the decomposition of Lesson 16.

**Reading the result.** `r.status == "optimal"` with `r.gap_certified`
True is a certificate: `r.bound` is a valid global bound and the gap closed.
`"feasible"` means an incumbent without a closed gap (check
`r.bound_valid` before using `r.bound` as a certificate). Local-only routes
report `"local_optimal"` — a stationary point with *no* global claim.

## 6. When spatial B&B fails

- **Loose bounds** → loose envelopes → loose relaxations → no pruning. Fix: bound tightening, OBBT, branching on bounds.
- **Symmetry** → many equivalent nodes. Fix: lex breaking (Lesson 29).
- **High dimension** → curse of dimensionality on box partitioning. Fix: structure (Benders, Lagrangian) when applicable.

## References
{cite:p}`Smith1999,Tawarmalani2005,Belotti2009,Belotti2013,Floudas1999,HorstTuy1996`.